In [ ]:
import signal
import wandb
import torch
import os 

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from hydra import compose, initialize

from codefiles.helpers import is_running_in_notebook  # for reloading modules instead of restarting kernel
if is_running_in_notebook():
    from codefiles import helpers
    from codefiles import architecture
    from codefiles import encoders
    from codefiles import transformer
    from codefiles.lightningdatamodules import mimic_symile
    from codefiles.lightningmodules import mimic
    import importlib
    importlib.reload(helpers)
    importlib.reload(architecture)
    importlib.reload(encoders)
    importlib.reload(transformer)
from codefiles.helpers import set_all_seeds, signal_handler, build_model
from codefiles.lightningmodules.mimic import MIMIC_Lightning_Module
from codefiles.lightningdatamodules.mimic_symile import MIMIC_Symile_Datamodule

os.environ["WANDB_SILENT"] = "true"
torch.set_float32_matmul_precision("high")

def main(cfg) -> None:
    set_all_seeds(seed=cfg.seed)
    wandb.init(
        project=cfg.wandb.project,
        config={key: value for key, value in cfg.items()},
    )

    model = build_model()
    lightningmodule = MIMIC_Lightning_Module(
        model=model
    )
    datamodule = MIMIC_Symile_Datamodule(
        batch_size=cfg.batch_size,
        seed=cfg.seed,
        #missing=,
        #variant=
    )

    trainer = pl.Trainer(
        logger=WandbLogger(project=cfg.wandb.project, dir="wandb/"),
        log_every_n_steps=1,
        accelerator='gpu',
        devices=1,
        max_epochs=50,
        # precision="bf16-mixed"
    )

    trainer.fit(lightningmodule, datamodule)
    wandb.finish()

if __name__ == "__main__":
    CONFIG_NAME = "config"
    signal.signal(signal.SIGINT, signal_handler)
    with initialize(version_base="1.1", config_path="config"):
        cfg = compose(config_name=f"{CONFIG_NAME}")

    # cfg = {key: value for key, value in cfg.items()}
    main(cfg)

Seed set to 420


{'modelname': {'optimizer': {'learning_rate': 3e-05, 'warmup_steps': 5, 'weight_decay': 0.0}, 'head_transformer': {'d_model': 512, 'num_layers': 2, 'dim_feedforward': 128, 'nhead': 4, 'dropout': 0.0}}, 'dataset': 'mimic_symile', 'batch_size': 512, 'seed': 420, 'wandb': {'project': 'simple_mml_baseline'}}
dict_keys(['modelname', 'dataset', 'batch_size', 'seed', 'wandb'])


/sc-projects/sc-proj-ukb-cvd/environments/mml/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:208: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/sc-projects/sc-proj-ukb-cvd/environments/mml/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /sc-projects/sc-proj-ukb-cvd/environments/mml/lib/py ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/sc-projects/sc-proj-ukb-cvd/environments/mml/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:44: Attribute 'model' removed from hparams because it cannot be pickled. You can suppress this warning by setting 

total_samples: 5812 / 5812
no_missing: 5812 / 5812
modality_0_missing: 0 / 5812
modality_1_missing: 0 / 5812
modality_2_missing: 0 / 5812


/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/codefiles/datasets/mimic_symile.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.ecgs = torch.load(f"{d

...


  | Name               | Type                         | Params | Mode 
----------------------------------------------------------------------------
0 | model              | Multimodal_Architecture      | 100 M  | train
1 | loss               | WeightedNaNBCEWithLogitsLoss | 0      | train
2 | acc_train          | MulticlassAccuracy           | 0      | train
3 | metric_train_macro | NaNMultilabelAUROC           | 0      | train
4 | metric_val_macro   | NaNMultilabelAUROC           | 0      | train
5 | metric_test_macro  | NaNMultilabelAUROC           | 0      | train
6 | metric_train_micro | NaNMultilabelAUROC           | 0      | train
7 | metric_val_micro   | NaNMultilabelAUROC           | 0      | train
8 | metric_test_micro  | NaNMultilabelAUROC           | 0      | train
----------------------------------------------------------------------------
100 M     Trainable params
0         Non-trainable params
100 M     Total params
402.911   Total estimated model params size (MB)
287  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/codefiles/datasets/mimic_symile.py:116: UserWarning: Lab values are not normalized.
  warnings.warn("Lab values are not normalized.")
/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/codefiles/datasets/mimic_symile.py:116: UserWarning: Lab values are not normalized.
  warnings.warn("Lab values are not normalized.")
/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/codefiles/datasets/mimic_symile.py:116: UserWarning: Lab values are not normalized.
  warnings.warn("Lab values are not normalized.")
/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/codefiles/datasets/mimic_symile.py:116: UserWarning: Lab values are not normalized.
  warnings.warn("Lab values are not normalized.")
/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/codefiles/datasets/mimic_symile.py:116: UserWarning: Lab values are not normalized.
  warnings.warn("Lab values are not normalized.")
/sc-projects/sc-proj-ukb-

Training: |          | 0/? [00:00<?, ?it/s]

/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/codefiles/datasets/mimic_symile.py:116: UserWarning: Lab values are not normalized.
  warnings.warn("Lab values are not normalized.")
